# Quantum VQC+GAT — 3000 patches (Laptop RTX 5060)

**Upgrade from E2:** `max_patches 1024 → 3000` — full slide coverage

| Setting | E2 (done) | This run |
|---|---|---|
| max_patches | 1024 (34% slide) | **3000 (100% slide)** |
| Test AUC | 0.6851 | target: **0.73–0.78** |
| Time/epoch | ~30 min | ~90 min |
| Est. total | — | **~25–30 hrs** (overnight + day) |

**VRAM safe:** ~0.8 GB needed (8 GB available) — `batch_size=4` OK


In [1]:
# Cell 1 — Setup
import os, sys, time, json, gc
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score

# CUDA fragmentation fix — critical for 30-hour runs
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'

sys.path.insert(0, '..')
from pathq.model_v2   import QuantaPathV2
from pathq.dataset_v2 import get_loaders_from_features

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT     = Path('..').resolve()
FEAT_DIR = Path('data') / 'features_uni'
CKPT_DIR = ROOT / 'checkpoints'
OUT_DIR  = ROOT / 'outputs'
CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    torch.cuda.empty_cache()
print(f'CKPT : {CKPT_DIR}')

/home/kabi/.conda/envs/pathq/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[model_v2] Using Transformer for global branch
GPU  : NVIDIA GeForce RTX 5060 Laptop GPU
VRAM : 8.1 GB
CKPT : /home/kabi/PATHQ--Quantum-Digital-Pathology-for-Whole-Slide-Image-Analysis/checkpoints


In [2]:
# Cell 2 — Load data (3000 patches — full slide)
train_loader, val_loader, test_loader = get_loaders_from_features(
    features_dir = FEAT_DIR,
    batch_size   = 2,      # safe at 3000p (VRAM ~0.8 GB)
    k            = 8,
    seed         = 42,
    max_patches  = 3000,
)
print(f'max_patches : 3000  (full slide)')
print(f'Train : {len(train_loader)} batches')
print(f'Val   : {len(val_loader)} batches')
print(f'Test  : {len(test_loader)} batches')
print()
print('Time estimate:')
print('  ~90 min/epoch  |  patience=8  |  max 40 epochs')
print('  Early stop ~ep16-20  →  ~24-30 hours total')
print('  Auto-saves every epoch — safe to interrupt & resume')

  Skipped 112 unlabeled file(s) (e.g. test_*) — keeping 221 labeled slides
Split: train=154 (pos=77) val=33 (pos=17) test=34 (pos=17)
max_patches : 3000  (full slide)
Train : 77 batches
Val   : 17 batches
Test  : 17 batches

Time estimate:
  ~90 min/epoch  |  patience=8  |  max 40 epochs
  Early stop ~ep16-20  →  ~24-30 hours total
  Auto-saves every epoch — safe to interrupt & resume


In [3]:
# Cell 3 — Training functions
SEP  = '═' * 68
DASH = '─' * 68

def train_one(model, loader, optimizer, device):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        torch.cuda.empty_cache()          # prevent CUDA fragmentation
        logits, _ = model(batch)
        loss      = F.cross_entropy(logits, batch.y.view(-1), label_smoothing=0.1)
        loss_val  = loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        del logits, loss
        torch.cuda.empty_cache()
        total += loss_val; n += 1
    return total / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    probs, labels, tl, n = [], [], 0.0, 0
    for batch in loader:
        batch     = batch.to(device)
        logits, _ = model(batch)
        tl       += F.cross_entropy(logits, batch.y.view(-1)).item()
        probs.extend(torch.softmax(logits, 1)[:, 1].cpu().tolist())
        labels.extend(batch.y.view(-1).cpu().tolist())
        torch.cuda.empty_cache()
        n += 1
    p, l  = np.array(probs), np.array(labels)
    preds = (p >= 0.5).astype(int)
    auc   = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
    f1    = f1_score(l, preds, zero_division=0)
    tp = int(((preds==1)&(l==1)).sum())
    fn = int(((preds==0)&(l==1)).sum())
    tn = int(((preds==0)&(l==0)).sum())
    fp = int(((preds==1)&(l==0)).sum())
    return {
        'auc'        : round(auc, 6),
        'f1'         : round(f1, 6),
        'loss'       : round(tl / max(n, 1), 6),
        'sensitivity': round(tp / max(tp+fn, 1), 4),
        'specificity': round(tn / max(tn+fp, 1), 4),
    }


def run_quantum_3000(model, tr, va, te, device,
                     ckpt_best, ckpt_latest,
                     epochs=40, lr=3e-5, patience=8):

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-2,
    )
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-7
    )
    plateau_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-7
    )

    best_auc, pat, start = 0.0, 0, 1

    if Path(ckpt_latest).exists():
        try:
            ck = torch.load(ckpt_latest, weights_only=False)
            model.load_state_dict(ck['model_state'])
            start    = ck['epoch'] + 1
            best_auc = ck['best_auc']
            for _ in range(start - 1): cosine_sched.step()
            print(f'Resumed from epoch {start-1}  best_auc={best_auc:.4f}')
        except Exception as e:
            print(f'Checkpoint incompatible — starting fresh. ({e})')
            start, best_auc = 1, 0.0
    else:
        print('No checkpoint — starting fresh.')

    print()
    print(SEP)
    print(' Quantum VQC+GAT — max_patches=3000  |  RTX 5060 laptop')
    print(' dropout=0.5 | weight_decay=1e-2 | label_smoothing=0.1')
    print(f' patience={patience} | epochs={epochs} | lr={lr}')
    print(SEP)
    print(f' {"Ep":>3} {"TrL":>8} {"VaL":>8} {"VaAUC":>7} '
          f'{"VaF1":>7} {"LR":>9} {"Min":>6}')
    print(DASH)

    for ep in range(start, epochs + 1):
        t0  = time.time()
        tl  = train_one(model, tr, optimizer, device)
        vm  = evaluate(model, va, device)

        cosine_sched.step()
        plateau_sched.step(vm['auc'])
        current_lr = optimizer.param_groups[0]['lr']
        flag       = ''

        if vm['auc'] > best_auc:
            best_auc = vm['auc']; pat = 0; flag = '*'
            torch.save({'model_state': model.state_dict(),
                        'epoch': ep, 'best_auc': best_auc}, ckpt_best)
        else:
            pat += 1

        torch.save({'model_state': model.state_dict(),
                    'epoch': ep, 'best_auc': best_auc}, ckpt_latest)

        ow   = ' ⚠overfit' if vm['loss'] > tl * 2.5 else ''
        mins = (time.time() - t0) / 60
        eta  = mins * (epochs - ep)
        print(f' {ep:>3} {tl:>8.4f} {vm["loss"]:>8.4f} '
              f'{vm["auc"]:>7.4f} {vm["f1"]:>7.4f} '
              f'{current_lr:>9.2e} {mins:>5.1f}m {flag}{ow}')

        if pat >= patience:
            print(f'\n Early stop ep {ep} — patience={patience}')
            break

        torch.cuda.empty_cache(); gc.collect()

    ck = torch.load(ckpt_best, weights_only=False)
    model.load_state_dict(ck['model_state'])
    tm = evaluate(model, te, device)

    print(DASH)
    print(f' Best val AUC : {best_auc:.4f}')
    print(f' Test AUC     : {tm["auc"]:.4f}')
    print(f' F1           : {tm["f1"]:.4f}')
    print(f' Sensitivity  : {tm["sensitivity"]:.4f}')
    print(f' Specificity  : {tm["specificity"]:.4f}')
    print(f' Val→Test gap : {tm["auc"]-best_auc:+.4f}')
    print(SEP)

    return {**tm, 'val_auc': best_auc,
            'gap': round(tm['auc'] - best_auc, 6)}

print('Functions loaded ✓')

Functions loaded ✓


In [4]:
# Cell 4 — Run Quantum 3000p
CKPT_BEST   = str(CKPT_DIR / 'E2_quantum_3000_best.pth')
CKPT_LATEST = str(CKPT_DIR / 'E2_quantum_3000_latest.pth')

model = QuantaPathV2(
    use_vqc    = True,
    n_qubits   = 3,
    vqc_layers = 2,
    in_dim     = 1040,
).to(DEVICE)

n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params : {n_p:,}')
print(f'VQC out_dim      : {model.vqc.out_dim}  (64)')
print()
print('Upgrade vs E2 (1024p):')
print('  max_patches : 3000  (was 1024 — 3x more tissue coverage)')
print('  batch_size  : 4     (VRAM safe — only ~0.8 GB needed)')
print('  All other settings unchanged')
print()

# OOM fallback: if VRAM error, interrupt and change batch_size=4 → 2 in Cell 2
result = run_quantum_3000(
    model,
    train_loader, val_loader, test_loader,
    DEVICE,
    ckpt_best   = CKPT_BEST,
    ckpt_latest = CKPT_LATEST,
    epochs      = 40,
    lr          = 3e-5,
    patience    = 8,
)

with open(OUT_DIR / 'E2_quantum_3000_result.json', 'w') as f:
    json.dump({
        'experiment'   : 'quantum_VQC_3000patches',
        'use_vqc'      : True,
        'n_qubits'     : 3,
        'vqc_layers'   : 2,
        'max_patches'  : 3000,
        'dropout'      : 0.5,
        'weight_decay' : 1e-2,
        'label_smooth' : 0.1,
        'patience'     : 8,
        **result,
    }, f, indent=2)

print('\nSaved: outputs/E2_quantum_3000_result.json')
print('Next  : run week7_ensemble.ipynb (update MAX_PATCHES=3000)')

[VQC] lightning.gpu  3q 2L — batched + re-uploading
QuantaPathV2: use_vqc=True, trainable=1,114,577
Trainable params : 1,114,577
VQC out_dim      : 64  (64)

Upgrade vs E2 (1024p):
  max_patches : 3000  (was 1024 — 3x more tissue coverage)
  batch_size  : 4     (VRAM safe — only ~0.8 GB needed)
  All other settings unchanged

Resumed from epoch 2  best_auc=0.6140

════════════════════════════════════════════════════════════════════
 Quantum VQC+GAT — max_patches=3000  |  RTX 5060 laptop
 dropout=0.5 | weight_decay=1e-2 | label_smoothing=0.1
 patience=8 | epochs=40 | lr=3e-05
════════════════════════════════════════════════════════════════════
  Ep      TrL      VaL   VaAUC    VaF1        LR    Min
────────────────────────────────────────────────────────────────────


/tmp/ipykernel_201651/3686125689.py:76: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  for _ in range(start - 1): cosine_sched.step()


   3   0.6764   0.6612  0.6471  0.5714  2.96e-05  82.6m *
   4   0.6582   0.6924  0.6140  0.5714  2.93e-05  82.9m 
   5   0.6682   0.7017  0.6691  0.5806  2.89e-05  82.7m *
   6   0.6621   0.7367  0.6912  0.5517  2.84e-05  83.5m *
   7   0.6387   0.7840  0.6581  0.5517  2.78e-05  82.9m 
   8   0.6342   0.8828  0.6801  0.5333  2.71e-05  82.5m 
   9   0.6298   0.9443  0.6654  0.6471  2.64e-05  82.7m 
  10   0.6179   1.0851  0.6691  0.5000  1.28e-05  82.9m 
  11   0.5747   0.9648  0.6691  0.5517  1.24e-05  82.7m 
  12   0.5692   1.0059  0.6728  0.6000  1.19e-05  82.7m 
  13   0.5777   1.0293  0.6544  0.6452  1.14e-05  82.6m 
  14   0.5887   0.9604  0.6691  0.5000  5.46e-06  82.6m 

 Early stop ep 14 — patience=8
────────────────────────────────────────────────────────────────────
 Best val AUC : 0.6912
 Test AUC     : 0.6817
 F1           : 0.6207
 Sensitivity  : 0.5294
 Specificity  : 0.8235
 Val→Test gap : -0.0095
════════════════════════════════════════════════════════════════════

Sav

In [5]:
# Cell 5 — Compare: 1024p vs 3000p
import json
from pathlib import Path

OUT = Path('..') / 'outputs'
SEP = '=' * 65

def load(fname):
    try:
        with open(OUT / fname) as f: return json.load(f)
    except: return None

e2      = load('E2_result.json')
e2_3000 = load('E2_quantum_3000_result.json')
e3_3000 = load('E3_classical_3000_result.json')

print()
print(SEP)
print(' QUANTUM — 1024p vs 3000p  (clean 221 slides)')
print(SEP)
print(f' {"":<30} {"Val AUC":>8} {"Test AUC":>9} {"F1":>7} {"Gap":>8}')
print(' ' + '-'*60)

for label, r in [
    ('E2 Quantum VQC+GAT (1024p)', e2),
    ('E2 Quantum VQC+GAT (3000p)', e2_3000),
    ('E3 Classical GAT   (3000p)', e3_3000),
]:
    if r:
        va  = r.get('val_auc', 0)
        ta  = r.get('auc', 0)
        f1  = r.get('f1', 0)
        gap = r.get('gap', ta - va)
        print(f' {label:<30} {va:>8.4f} {ta:>9.4f} {f1:>7.4f} {gap:>+8.4f}')
    else:
        print(f' {label:<30} {"pending":>8}')

print(SEP)
if e2 and e2_3000:
    gain = e2_3000['auc'] - e2['auc']
    print(f' 3000p gain vs 1024p: {gain:+.4f} AUC')


 QUANTUM — 1024p vs 3000p  (clean 221 slides)
                                 Val AUC  Test AUC      F1      Gap
 ------------------------------------------------------------
 E2 Quantum VQC+GAT (1024p)       0.6507    0.6851  0.6875  +0.0344
 E2 Quantum VQC+GAT (3000p)       0.6912    0.6817  0.6207  -0.0095
 E3 Classical GAT   (3000p)       0.6581    0.6920  0.7097  +0.0340
 3000p gain vs 1024p: -0.0035 AUC
